# Explainable AI (XAI) - WESAD

Questo notebook implementa l'estrazione e la visualizzazione delle mappe di attenzione per il dataset WESAD.
Carica i pesi del modello fine-tunato, carica il dataset di test WESAD, effettua una predizione su un paziente e genera le heatmap dell'attenzione (Raw Attention, Attention Rollout e Gradient-weighted Attention).

In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import argparse
import re

# 1. Setup dell'ambiente di lavoro su Kaggle
WORK_DIR = '/kaggle/working/e4selflearning'
if not os.path.exists(WORK_DIR):
    !git clone https://github.com/april-tools/e4selflearning.git {WORK_DIR}

if WORK_DIR not in sys.path:
    sys.path.insert(0, os.path.join(WORK_DIR, 'src'))

os.chdir(WORK_DIR)

from torch.utils.data import DataLoader
from timebase.models.models import get_models
from timebase.data.dataset import ClassificationDataset

class TransformerXAI:
    """
    Classe helper per l'estrazione e l'analisi delle mappe di attenzione (XAI)
    nei modelli basati su Transformer. Utilizza i PyTorch hooks per catturare 
    le matrici di attenzione e i relativi gradienti durante il passaggi di forward e backward.
    """
    def __init__(self, model):
        self.model = model
        self.model.eval()
        
        self.attention_weights = []
        self.gradients = []
        self.hooks = []
        
        self._register_hooks()
        
    def _register_hooks(self):
        def forward_hook_fn(module, input, output):
            # Salva i pesi di attenzione durante il forward pass
            self.attention_weights.append(output)
            
            # Registra un hook sul gradiente per il backward pass
            if output.requires_grad:
                output.register_hook(lambda grad: self.gradients.append(grad))
                
        # Registra l'hook su tutti i moduli Softmax (attenzione) del modello
        for name, module in self.model.named_modules():
            if isinstance(module, torch.nn.Softmax):
                hook = module.register_forward_hook(forward_hook_fn)
                self.hooks.append(hook)
                
    def clear(self):
        """Pulisce i pesi e i gradienti accumulati per una nuova predizione."""
        self.attention_weights = []
        self.gradients = []
        
    def remove_hooks(self):
        """Rimuove tutti gli hook registrati per liberare risorse."""
        for hook in self.hooks:
            hook.remove()
        self.hooks = []

    def get_raw_attention(self, layer_idx=-1):
        """Estrae l'attenzione grezza di uno specifico livello (default: ultimo)."""
        return self.attention_weights[layer_idx].detach()
        
    def compute_rollout(self):
        """
        Calcola l'Attention Rollout per tracciare il flusso dell'attenzione 
        attraverso i vari layer del Transformer (Abnar & Zuidema, 2020).
        """
        rollout = None
        for attn in self.attention_weights:
            # Media sulle teste di attenzione
            attn_mean = attn.mean(dim=1).detach()
            
            # Aggiunge la matrice identità per considerare le connessioni residuali (self-attention)
            identity = torch.eye(attn.size(-1)).to(attn.device)
            a = attn_mean + identity
            
            # Normalizzazione
            a = a / a.sum(dim=-1, keepdim=True)
            
            if rollout is None:
                rollout = a
            else:
                rollout = torch.matmul(a, rollout)
                
        return rollout

class WESAD_ClassificationDataset(ClassificationDataset):
    def session_id(self, filename: str) -> int:
        m = re.search(r'[sS](\d+)', filename)
        return int(m.group(1)) if m else 0

print("Setup completato con successo.")

In [ ]:
# 2. Caricamento del Modello Fine-Tunato e dei Dati WESAD
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WES_DIR = None
W_FILE = None

for r, d, f in os.walk('/kaggle/input'):
    for file in f:
        if file == "metadata.pkl" and "wesad" in r.lower():
            WES_DIR = r
        if file == "finetuned_model.pt":
            W_FILE = os.path.join(r, file)

if not WES_DIR or not W_FILE:
    print("[!] ERRORE: Dataset o pesi WESAD non trovati in /kaggle/input.")
else:
    with open(os.path.join(WES_DIR, "metadata.pkl"), "rb") as f:
        meta = pickle.load(f)
    
    # Definizione Namespace dei parametri per il modello
    args = argparse.Namespace()
    args.num_train_subjects = 10
    args.device = device
    args.task_mode = 1
    args.e4selflearning = True
    args.verbose = 0
    args.batch_size = 1
    args.output_dir = "."
    args.use_wandb = False
    args.dataset = WES_DIR
    
    # Configurazione del modello (Encoder + Representation Module)
    args.pretext_task = "masked_prediction"
    args.encoder_type = "resnet"
    args.representation_module = "transformer"
    args.mask_ratio = 0.5
    args.emb_dim = 64
    args.emb_num_filters = 4
    args.emb_type = 1
    args.mlp_dim = 64
    args.num_blocks = 3
    args.num_heads = 3
    args.num_units = 64
    
    # Dropout e regolarizzazione
    args.a_dropout = 0.0
    args.m_dropout = 0.0
    args.dropout = 0.0
    args.drop_path = 0.0
    args.disable_bias = False
    args.split_mode = 0
    args.scaling_mode = 2
    args.critic_score_lambda = 0.0
    
    args.ds_info = meta["ds_info"]
    args.input_shapes = {
        c: [512 * args.ds_info["channel_freq"][c]] 
        for c in ['ACC_x', 'ACC_y', 'ACC_z', 'BVP', 'EDA', 'TEMP']
    }
    
    # Caricamento o generazione statistiche per la normalizzazione dei segnali
    stats_path = os.path.join(WES_DIR, "stats.pkl")
    stats = None
    if os.path.exists(stats_path):
        with open(stats_path, "rb") as f:
            sl = pickle.load(f)
            stats = sl if "BVP" in sl else sl[list(sl.keys())[0]]
            
    if stats is None or "BVP" not in stats:
        print("[WARNING] stats.pkl non trovato o non valido. Uso stats di default per normalizzazione.")
        stats = {
            c: {'min': -10, 'max': 10, 'mean': 0, 'std': 1, 'median': 0, 'iqr': 1} 
            for c in ['ACC_x', 'ACC_y', 'ACC_z', 'BVP', 'EDA', 'TEMP']
        }
    
    # Divisione soggetti per il Test Split
    all_paths = np.array([
        os.path.join(WES_DIR, p.replace("\\", "/").split("/")[-2], p.replace("\\", "/").split("/")[-1]) 
        for p in meta["sessions_paths"]
    ])
    labels = {k: np.array(v) for k, v in meta["sessions_labels"].items()}
    
    te_subs = ["S13", "S14", "S15", "S16", "S17"]
    m_te = np.array([any(s in p.upper() for s in te_subs) for p in all_paths])
    if not m_te.any():
        m_te = np.ones(len(all_paths), dtype=bool)
    
    ds_te = WESAD_ClassificationDataset(
        args=args, 
        filenames=all_paths[m_te], 
        labels={k: v[m_te] for k, v in labels.items()}, 
        rec_ids=all_paths[m_te], 
        stats=stats, 
        recording_id_str_to_num={p: i for i, p in enumerate(np.unique(all_paths[m_te]))}
    )
    loader_te = DataLoader(ds_te, batch_size=1, shuffle=True)
    
    # Inizializzazione modello e caricamento pesi
    classifier, _ = get_models(args, summary=None)
    classifier.load_state_dict(torch.load(W_FILE, map_location=device))
    classifier.to(device)
    print("[OK] Modello e Dati WESAD caricati correttamente.")

In [ ]:
# 3. Estrazione Mappe di Spiegabilità (XAI) e Generazione Plot
if 'classifier' in locals() and 'loader_te' in locals():
    # Instanzia lo strumento di spiegabilità
    xai_tool = TransformerXAI(classifier)
    
    # STEP 1 & 2: Loop per trovare il caso "più favorevole" (predizione corretta dello Stress con EDA attiva nel crop)
    print("Ricerca di un campione con predizione corretta ed EDA attiva nella finestra di 60s...")
    found_sample = False
    for batch in loader_te:
        xai_tool.clear() # Pulisce prima del forward pass per evitare l'IndexError delle liste di hook
        inputs = {k: v.float().to(device) for k, v in batch["data"].items()}
        true_label = batch["target"][0].item()
        
        # Forward pass per predizione
        out, _ = classifier(inputs)
        p_prob = torch.sigmoid(out).item()
        pred_label = 1.0 if p_prob >= 0.5 else 0.0
        
        # Cerchiamo un campione Stress (1.0) predetto correttamente come Stress (1.0) con alta confidenza
        if true_label == 1.0 and pred_label == 1.0 and p_prob >= 0.7:
            # Calcolo dei gradienti per questo campione
            xai_tool.model.zero_grad()
            out.sum().backward(retain_graph=True)
            
            # Calcolo g_1d_stress
            gradients_tensor = xai_tool.gradients[0][0]
            g = gradients_tensor.mean(dim=0).detach().cpu().numpy()
            a = xai_tool.attention_weights[-1][0].mean(dim=0).detach().cpu().numpy()
            ga_stress = np.maximum(0, g * a)
            np.fill_diagonal(ga_stress, 0)
            ga_stress[:, 0] = 0
            
            v1d = np.sqrt(np.maximum(0, ga_stress.mean(axis=0)))
            g_1d_stress = (v1d - v1d.min()) / (v1d.max() - v1d.min() + 1e-8)
            
            # Trova il picco del segnale EDA e centra la finestra di 60 secondi su di esso
            eda = inputs['EDA'][0].detach().cpu().numpy().flatten()
            eda_max_idx = np.argmax(eda)
            seq_len_sec = len(g_1d_stress)
            fs_eda = len(eda) / seq_len_sec
            max_idx = int(eda_max_idx / fs_eda)
            
            window_size = 60
            if seq_len_sec > window_size:
                start_sec = max(0, max_idx - window_size // 2)
                end_sec = min(seq_len_sec, start_sec + window_size)
                if end_sec == seq_len_sec:
                    start_sec = max(0, end_sec - window_size)
            else:
                start_sec = 0
                end_sec = seq_len_sec
                
            # Crop dell'EDA per verificarne il range nella finestra selezionata
            eda_crop = eda[int(start_sec * fs_eda):int(end_sec * fs_eda)]
            eda_range = eda_crop.max() - eda_crop.min()
            
            if eda_range > 0.05:
                print(f"[OK] Campione Stress corretto trovato! Confidenza: {p_prob:.2f} | EDA Crop Range: {eda_range:.3f} | Centrato sul secondo {max_idx}")
                found_sample = True
                break
            
    if not found_sample:
        # Secondo tentativo: una predizione qualunque corretta con EDA attiva nella finestra
        print("[INFO] Secondo tentativo: ricerca di un campione corretto generico...")
        for batch in loader_te:
            xai_tool.clear() # Pulisce prima del forward pass
            inputs = {k: v.float().to(device) for k, v in batch["data"].items()}
            true_label = batch["target"][0].item()
            out, _ = classifier(inputs)
            p_prob = torch.sigmoid(out).item()
            pred_label = 1.0 if p_prob >= 0.5 else 0.0
            
            if true_label == pred_label:
                xai_tool.model.zero_grad()
                out.sum().backward(retain_graph=True)
                
                gradients_tensor = xai_tool.gradients[0][0]
                g = gradients_tensor.mean(dim=0).detach().cpu().numpy()
                a = xai_tool.attention_weights[-1][0].mean(dim=0).detach().cpu().numpy()
                ga_stress = np.maximum(0, g * a)
                np.fill_diagonal(ga_stress, 0)
                ga_stress[:, 0] = 0
                
                v1d = np.sqrt(np.maximum(0, ga_stress.mean(axis=0)))
                g_1d_stress = (v1d - v1d.min()) / (v1d.max() - v1d.min() + 1e-8)
                
                # Trova il picco del segnale EDA e centra la finestra di 60 secondi su di esso
                eda = inputs['EDA'][0].detach().cpu().numpy().flatten()
                eda_max_idx = np.argmax(eda)
                seq_len_sec = len(g_1d_stress)
                fs_eda = len(eda) / seq_len_sec
                max_idx = int(eda_max_idx / fs_eda)
                
                window_size = 60
                if seq_len_sec > window_size:
                    start_sec = max(0, max_idx - window_size // 2)
                    end_sec = min(seq_len_sec, start_sec + window_size)
                    if end_sec == seq_len_sec:
                        start_sec = max(0, end_sec - window_size)
                else:
                    start_sec = 0
                    end_sec = seq_len_sec
                    
                eda_crop = eda[int(start_sec * fs_eda):int(end_sec * fs_eda)]
                eda_range = eda_crop.max() - eda_crop.min()
                
                if eda_range > 0.05:
                    print(f"[OK] Campione corretto trovato (Label: {true_label}, Pred: {pred_label})! Confidenza: {p_prob:.2f} | EDA Crop Range: {eda_range:.3f}")
                    found_sample = True
                    break
                
    if not found_sample:
        # Fallback: primo campione del loader
        print("[WARNING] Nessun campione ideale con EDA attiva nella finestra trovato. Uso fallback.")
        batch = next(iter(loader_te))
        xai_tool.clear() # Pulisce prima del forward pass
        inputs = {k: v.float().to(device) for k, v in batch["data"].items()}
        true_label = batch["target"][0].item()
        out, _ = classifier(inputs)
        p_prob = torch.sigmoid(out).item()
        
        xai_tool.model.zero_grad()
        out.sum().backward(retain_graph=True)
    
    # 1. Estrazione Raw Attention (ultimo layer)
    raw_attn_tensor = xai_tool.get_raw_attention(-1)[0]
    raw = raw_attn_tensor.mean(dim=0).detach().cpu().numpy()
    np.fill_diagonal(raw, 0) # Rimuove l'auto-attenzione sulla diagonale
    
    # 2. Calcolo Attention Rollout (Abnar & Zuidema, 2020)
    rollout_tensor = xai_tool.compute_rollout()[0]
    roll = rollout_tensor.detach().cpu().numpy()
    np.fill_diagonal(roll, 0)
    
    # 3. Calcolo Gradient-weighted Attention (Attention x Gradient)
    gradients_tensor = xai_tool.gradients[0][0]
    g = gradients_tensor.mean(dim=0).detach().cpu().numpy()
    a = xai_tool.attention_weights[-1][0].mean(dim=0).detach().cpu().numpy()
    
    # Moltiplicazione punto a punto per Stress (gradienti positivi) e Non-Stress (gradienti negativi)
    ga_stress = np.maximum(0, g * a)
    ga_nonstress = np.maximum(0, -g * a)
    np.fill_diagonal(ga_stress, 0)
    np.fill_diagonal(ga_nonstress, 0)
    
    # Rimozione dell'attention sink (token iniziale/CLS/padding all'indice 0)
    raw[:, 0] = 0
    roll[:, 0] = 0
    ga_stress[:, 0] = 0
    ga_nonstress[:, 0] = 0
    
    # Helper function per elaborare i vettori 1D normalizzati
    def process(v):
        # Media lungo l'asse temporale per ottenere il profilo monodimensionale
        v1d = np.sqrt(np.maximum(0, v.mean(axis=0)))
        # Min-Max Normalization
        return (v1d - v1d.min()) / (v1d.max() - v1d.min() + 1e-8)
    
    r_1d = process(raw)
    ro_1d = process(roll)
    g_1d_stress = process(ga_stress)
    g_1d_nonstress = process(ga_nonstress)
    
    seq_len_sec = len(r_1d)
    
    # STEP 2: Trova il picco del segnale EDA e centra la finestra di 60 secondi su di esso
    eda_full = inputs['EDA'][0].detach().cpu().numpy().flatten()
    eda_max_idx = np.argmax(eda_full)
    fs_eda_temp = len(eda_full) / seq_len_sec
    max_idx = int(eda_max_idx / fs_eda_temp)
    
    window_size = 60
    if seq_len_sec > window_size:
        start_sec = max(0, max_idx - window_size // 2)
        end_sec = min(seq_len_sec, start_sec + window_size)
        if end_sec == seq_len_sec:
            start_sec = max(0, end_sec - window_size)
    else:
        start_sec = 0
        end_sec = seq_len_sec
        
    print(f"Finestra temporale selezionata: secondi {start_sec} - {end_sec} (Centrata sul picco EDA al secondo {max_idx})")
    
    # Slicing dell'attenzione alla finestra di 60 secondi
    r_1d_crop = r_1d[start_sec:end_sec]
    ro_1d_crop = ro_1d[start_sec:end_sec]
    g_1d_stress_crop = g_1d_stress[start_sec:end_sec]
    g_1d_nonstress_crop = g_1d_nonstress[start_sec:end_sec]
    
    # Helper per la normalizzazione dei segnali fisiologici
    def normalize_signal(sig):
        return (sig - sig.min()) / (sig.max() - sig.min() + 1e-8)
        
    # Caricamento segnali originari
    eda = inputs['EDA'][0].detach().cpu().numpy().flatten()
    bvp = inputs['BVP'][0].detach().cpu().numpy().flatten()
    temp = inputs['TEMP'][0].detach().cpu().numpy().flatten()
    
    acc_x = inputs['ACC_x'][0].detach().cpu().numpy().flatten()
    acc_y = inputs['ACC_y'][0].detach().cpu().numpy().flatten()
    acc_z = inputs['ACC_z'][0].detach().cpu().numpy().flatten()
    acc_mag = np.sqrt(acc_x**2 + acc_y**2 + acc_z**2)
    
    # Calcolo frequenze di campionamento
    fs_eda = len(eda) / seq_len_sec
    fs_bvp = len(bvp) / seq_len_sec
    fs_temp = len(temp) / seq_len_sec
    fs_acc = len(acc_mag) / seq_len_sec
    
    # Slicing dei segnali fisiologici alla finestra di 60 secondi
    eda_crop = eda[int(start_sec * fs_eda):int(end_sec * fs_eda)]
    bvp_crop = bvp[int(start_sec * fs_bvp):int(end_sec * fs_bvp)]
    temp_crop = temp[int(start_sec * fs_temp):int(end_sec * fs_temp)]
    acc_crop = acc_mag[int(start_sec * fs_acc):int(end_sec * fs_acc)]
    
    # Normalizzazione dei segnali ritagliati
    eda_norm = normalize_signal(eda_crop)
    bvp_norm = normalize_signal(bvp_crop)
    temp_norm = normalize_signal(temp_crop)
    acc_norm = normalize_signal(acc_crop)
    
    # Generazione assi x per i grafici
    plot_len_sec = end_sec - start_sec
    time_x = np.linspace(0, plot_len_sec, len(r_1d_crop))
    time_eda = np.linspace(0, plot_len_sec, len(eda_norm))
    time_bvp = np.linspace(0, plot_len_sec, len(bvp_norm))
    time_temp = np.linspace(0, plot_len_sec, len(temp_norm))
    time_acc = np.linspace(0, plot_len_sec, len(acc_norm))
    
    # DECIMAZIONE per una migliore visualizzazione dei segnali ad alta frequenza (BVP e ACC)
    dec_bvp = max(1, len(bvp_norm) // 480)
    time_bvp_dec = time_bvp[::dec_bvp]
    bvp_norm_dec = bvp_norm[::dec_bvp]
    
    dec_acc = max(1, len(acc_norm) // 480)
    time_acc_dec = time_acc[::dec_acc]
    acc_norm_dec = acc_norm[::dec_acc]
    
    # Creazione della cartella di output se non esiste
    os.makedirs('xai/output', exist_ok=True)
    dataset_name = "WESAD" if seq_len_sec == 512 else "HOSSEINI"
    
    # Imports per copiare file
    import shutil
    
    # Helper per styling legenda
    leg_args = dict(loc="upper left", facecolor="white", edgecolor="lightgrey", framealpha=0.9, labelcolor="black")
    
    # ----------------------------------------------------
    # FIGURA 1: GRAFICO UNIFICATO COMPLETO (Semplificato a 4 pannelli)
    # Contiene EDA, Attention Rollout, Raw Attention, e Gradient Importance
    # ----------------------------------------------------
    fig_full, axes_f = plt.subplots(4, 1, figsize=(12, 12), sharex=True)
    
    # 1. EDA
    axes_f[0].plot(time_eda, eda_norm, color="purple", label="EDA (Normalized)", linewidth=1.5)
    axes_f[0].set_ylabel("EDA")
    axes_f[0].legend(**leg_args)
    axes_f[0].set_title(f"Physiological Interpretability Map (Complete - {dataset_name})\nReal Label: {true_label} | Prediction: {1.0 if p_prob >= 0.5 else 0.0} (Confidence: {p_prob:.2f})")
    
    # 2. Raw Attention (Last Layer)
    axes_f[1].plot(time_x, r_1d_crop, color="blue", label="Raw Attention (Last Layer)", linewidth=1.5, linestyle="--")
    axes_f[1].fill_between(time_x, r_1d_crop, color="blue", alpha=0.15)
    axes_f[1].set_ylabel("Raw Attention")
    axes_f[1].legend(**leg_args)
    
    # 3. Attention Rollout
    axes_f[2].plot(time_x, ro_1d_crop, color="green", label="Attention Rollout", linewidth=2)
    axes_f[2].fill_between(time_x, ro_1d_crop, color="green", alpha=0.15)
    axes_f[2].set_ylabel("Attention Rollout")
    axes_f[2].legend(**leg_args)
    
    # 4. Gradient Importance (Stress & Non-Stress sovrapposti)
    axes_f[3].plot(time_x, g_1d_stress_crop, color="red", label="Grad Imp (Stress Class - positive logit)", linewidth=2)
    axes_f[3].fill_between(time_x, g_1d_stress_crop, color="red", alpha=0.15)
    axes_f[3].plot(time_x, g_1d_nonstress_crop, color="blue", label="Grad Imp (Non-Stress Class - negative logit)", linewidth=2)
    axes_f[3].fill_between(time_x, g_1d_nonstress_crop, color="blue", alpha=0.10)
    axes_f[3].set_ylabel("Grad Importance")
    axes_f[3].set_xlabel("Time (Seconds) inside selected 60s window")
    axes_f[3].legend(**leg_args)
    
    for ax in axes_f:
        ax.set_xlim(0, plot_len_sec)
        ax.grid(True, linestyle=":", alpha=0.6)
        
    plt.tight_layout()
    fig_f_path = f"xai/output/{dataset_name.lower()}_xai_combined_full.png"
    fig_full.savefig(fig_f_path, dpi=300, bbox_inches='tight')
    try:
        shutil.copy(fig_f_path, f"/kaggle/working/{dataset_name.lower()}_xai_combined_full.png")
        print(f"[KAGGLE] Salvato grafico unificato completo in: /kaggle/working/{dataset_name.lower()}_xai_combined_full.png")
    except Exception as e:
        pass
    plt.show()
    
    # ----------------------------------------------------
    # FIGURA 2: GRAFICO UNIFICATO OTTIMIZZATO (Semplificato a 4 pannelli)
    # Esclude BVP, TEMP e ACC. "Stacca" attention rollout e last layer.
    # ----------------------------------------------------
    fig_opt, axes_o = plt.subplots(4, 1, figsize=(12, 12), sharex=True)
    
    # 1. EDA
    axes_o[0].plot(time_eda, eda_norm, color="purple", label="EDA (Normalized)", linewidth=1.5)
    axes_o[0].set_ylabel("EDA")
    axes_o[0].legend(**leg_args)
    axes_o[0].set_title(f"Physiological Interpretability Map (Optimized - {dataset_name})\nReal Label: {true_label} | Prediction: {1.0 if p_prob >= 0.5 else 0.0} (Confidence: {p_prob:.2f})")
    
    # 2. Raw Attention (Last Layer)
    axes_o[1].plot(time_x, r_1d_crop, color="blue", label="Raw Attention (Last Layer)", linewidth=1.5, linestyle="--")
    axes_o[1].fill_between(time_x, r_1d_crop, color="blue", alpha=0.15)
    axes_o[1].set_ylabel("Raw Attention")
    axes_o[1].legend(**leg_args)
    
    # 3. Attention Rollout
    axes_o[2].plot(time_x, ro_1d_crop, color="green", label="Attention Rollout", linewidth=2)
    axes_o[2].fill_between(time_x, ro_1d_crop, color="green", alpha=0.15)
    axes_o[2].set_ylabel("Attention Rollout")
    axes_o[2].legend(**leg_args)
    
    # 4. Gradient Importance (Stress & Non-Stress sovrapposti)
    axes_o[3].plot(time_x, g_1d_stress_crop, color="red", label="Grad Imp (Stress Class - positive logit)", linewidth=2)
    axes_o[3].fill_between(time_x, g_1d_stress_crop, color="red", alpha=0.15)
    axes_o[3].plot(time_x, g_1d_nonstress_crop, color="blue", label="Grad Imp (Non-Stress Class - negative logit)", linewidth=2)
    axes_o[3].fill_between(time_x, g_1d_nonstress_crop, color="blue", alpha=0.10)
    axes_o[3].set_ylabel("Grad Importance")
    axes_o[3].set_xlabel("Time (Seconds) inside selected 60s window")
    axes_o[3].legend(**leg_args)
    
    for ax in axes_o:
        ax.set_xlim(0, plot_len_sec)
        ax.grid(True, linestyle=":", alpha=0.6)
        
    plt.tight_layout()
    fig_o_path = f"xai/output/{dataset_name.lower()}_xai_combined_optimized.png"
    fig_opt.savefig(fig_o_path, dpi=300, bbox_inches='tight')
    try:
        shutil.copy(fig_o_path, f"/kaggle/working/{dataset_name.lower()}_xai_combined_optimized.png")
        print(f"[KAGGLE] Salvato grafico unificato ottimizzato in: /kaggle/working/{dataset_name.lower()}_xai_combined_optimized.png")
    except Exception as e:
        pass
    plt.show()
    
    xai_tool.model.zero_grad()
